# ViT Assignment



## PART 1 Setup Directory

Run this cell first to mount Google Drive and set the working directory to your **CIS515** folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Set working directory to CIS515 folder on Google Drive
COLAB_DIR = '/content/drive/MyDrive/CIS515'
os.makedirs(COLAB_DIR, exist_ok=True)
os.chdir(COLAB_DIR)

print(f"Working directory: {os.getcwd()}")
print("Files in CIS515 folder:")
print(os.listdir(COLAB_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/CIS515
Files in CIS515 folder:
['competition_data']


In [3]:
# Install dependencies
!pip install transformers datasets optuna torch torchvision accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 31.9 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset, concatenate_datasets

print("Loading raw images from HuggingFace...")
raw = load_dataset("dpdl-benchmark/oxford_flowers102")

all_data = concatenate_datasets([raw["train"], raw["validation"], raw["test"]])

Loading raw images from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/test-00000-of-00006.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1020 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6149 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1020 [00:00<?, ? examples/s]

## Import Libraries and setup GPU

In [5]:
import torch
import optuna
import numpy as np
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    get_scheduler,
)


N_CLASSES  = 102
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


##Load & Preprocess Dataset

In [6]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset
from datasets import load_dataset, concatenate_datasets

################ PART 1 ###############
#(1) Enter Your Code Here
#(1.2) load ViT Image processor
# Load pretrained ViT
MODEL_NAME = "google/vit-base-patch16-224"
processor  = ViTImageProcessor.from_pretrained(MODEL_NAME)
########################################

COMPETITION_DIR = "/content/drive/MyDrive/CIS515/competition_data/public"

from torch.utils.data import Dataset

class FlowerDataset(Dataset):
    def __init__(self, csv_path, all_data, processor, has_labels=True):
        self.df         = pd.read_csv(csv_path)
        self.indices    = self.df["image_id"].str.replace("img_", "").astype(int).tolist()
        self.all_data   = all_data
        self.processor  = processor
        self.has_labels = has_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image = self.all_data[self.indices[idx]]["image"].convert("RGB")
        enc   = self.processor(images=image, return_tensors="pt")
        pv    = enc["pixel_values"].squeeze(0)   # (3, 224, 224)

        if self.has_labels:
            label = int(self.df["label"].iloc[idx])
            return pv, torch.tensor(label)
        return (pv,)


train_ds = FlowerDataset(f"{COMPETITION_DIR}/train.csv", all_data, processor)
val_ds   = FlowerDataset(f"{COMPETITION_DIR}/val.csv",   all_data, processor)
test_ds  = FlowerDataset(f"{COMPETITION_DIR}/test_public.csv", all_data, processor, has_labels=False)

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]


Train: 4913 | Val: 1228 | Test: 1228


## Training Function

In [7]:
import copy
    # Starter values
BATCH_SIZE   = 32
N_UNFREEZE   = 10  # allows to unfreeze last N_UNFREEZE encoder blocks
WEIGHT_DECAY = 0.01
LR=1e-2
WARMUP_RATIO=0.0
SCHEDULER_TYPE="constant_with_warmup"
EPOCHS     = 3
N_TRIALS   = 15

def train_and_evaluate(


    lr: float = LR,
    warmup_ratio: float = WARMUP_RATIO,
    scheduler_type: str = SCHEDULER_TYPE,

    n_unfreeze: int     = N_UNFREEZE,
    weight_decay: float = WEIGHT_DECAY,
    batch_size: int     = BATCH_SIZE,


    trial               = None,


):


    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=64)



    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=N_CLASSES,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)


    for param in model.vit.parameters():
        param.requires_grad = False


    for param in model.classifier.parameters():
        param.requires_grad = True



    if n_unfreeze > 0:
        for block in model.vit.encoder.layer[-n_unfreeze:]:
            for param in block.parameters():
                param.requires_grad = True

    for param in model.vit.layernorm.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )

    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * warmup_ratio)

    scheduler = get_scheduler(
        scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_val_acc    = 0.0
    best_model_state = None

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            pixel_values, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss         = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
                preds   = model(pixel_values=pixel_values).logits.argmax(dim=-1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_acc = correct / total
        if val_acc >= best_val_acc:
            best_val_acc     = val_acc
            best_model_state = copy.deepcopy(model.state_dict())

        print(f"    Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.3f} | val_acc={val_acc:.3f}")


        if trial is not None:
            trial.report(val_acc, epoch)

            trial.report(val_acc, epoch)
            if trial.should_prune():
                print("Trial pruned.")
                raise optuna.exceptions.TrialPruned()

    model.load_state_dict(best_model_state)

    return  best_val_acc, model

## PART 2: Baseline Model Performance

In [8]:
print("========================================")
print("BASELINE hyperparameters")
print("========================================")


base_acc, _ = train_and_evaluate(
    lr=LR,
    warmup_ratio=WARMUP_RATIO,
    scheduler_type=SCHEDULER_TYPE,
)
print(f"\nBASELINE config val accuracy: {base_acc*100:.1f}%")

BASELINE hyperparameters


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 70,958,694 / 85,877,094 (82.6%)
    Epoch 1/3 | loss=4.795 | val_acc=0.016
    Epoch 2/3 | loss=3.965 | val_acc=0.075
    Epoch 3/3 | loss=3.420 | val_acc=0.179

BASELINE config val accuracy: 17.9%


#PART 3 Run Optuna Study

In [ ]:
def objective(trial: optuna.Trial) -> float:

    # Existing hyperparameters (with updated ranges)
    lr             = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    warmup_ratio   = trial.suggest_float("warmup_ratio", 0.0, 0.3)
    scheduler_type = trial.suggest_categorical("scheduler_type",
                         ["linear", "cosine", "constant_with_warmup"])

    #(3.1) ENTER YOUR CODE HERE: EXPLORE ADDITIONAL HYPERPARAMETERS AND HYPERPARAMTER VALUES###
    n_unfreeze     = trial.suggest_int("n_unfreeze", 1, 12)
    weight_decay   = trial.suggest_float("weight_decay", 1e-4, 0.1, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [16, 32, 64])

    print(f"\nTrial {trial.number}: lr={lr:.2e}, warmup={warmup_ratio:.2f}, "
          f"sched={scheduler_type}, unfreeze={n_unfreeze}, "
          f"wd={weight_decay:.4f}, bs={batch_size}") #####PRINT HYPERPARAMETERS HERE##########

    val_acc, _ = train_and_evaluate(
        lr=lr, warmup_ratio=warmup_ratio, scheduler_type=scheduler_type,
        n_unfreeze=n_unfreeze, weight_decay=weight_decay, batch_size=batch_size,
        trial=trial, ######## (3.1) Update this function to use all hyperparameteres in trials
    )
    return val_acc


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)
study.optimize(objective, n_trials=N_TRIALS)

[I 2026-04-10 15:13:16,686] A new study created in memory with name: no-name-ab737c7a-1d2a-47ac-8f05-181398f7e60c



Trial 0: lr=5.61e-05, warmup=0.29, sched=linear, unfreeze=2, wd=0.0001, bs=16


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)


##Best Model vs Baseline Model Comparison

In [ ]:
print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"\nBASE  hyperparams → val accuracy: {base_acc*100:.1f}%")
print(f"BEST hyperparams → val accuracy: {study.best_value*100:.1f}%")
print(f"\nBest params found:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# PART 4 Final Evaluation

In [ ]:
best = study.best_params

#####################PART4#############
best_val_acc, best_model = train_and_evaluate(
    lr=best["lr"],
    warmup_ratio=best["warmup_ratio"],
    scheduler_type=best["scheduler_type"],

   ######ADD YOUR FINAL HYPERPARAMETERS HERE
)

print(f"\nFinal TEST accuracy: {best_val_acc*100:.1f}%")


##PART 5 Prediction on the Test Set
Saves your predictions CSV to `MyDrive/CIS515/competition_data/submissions/`.

In [ ]:

##################REPLACE WITH YOUR ASU ID###########
STUDENT_NAME       = "ksiamion-replacewithyourname"

In [ ]:
import pandas as pd
import os
######################PART 4 #########################

test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)


best_model.eval()
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch[0].to(DEVICE)  # TensorDataset returns tuples, not dicts
        preds = best_model(pixel_values=pixel_values).logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())


test_csv  = pd.read_csv(f"{COMPETITION_DIR}/test_public.csv")
image_ids = test_csv["image_id"].tolist()
submission = pd.DataFrame({"image_id": image_ids, "label": all_preds})

submission_path = f"/content/drive/MyDrive/CIS515/competition_data/submissions/{STUDENT_NAME}_public_.csv"

os.makedirs(os.path.dirname(submission_path), exist_ok=True)
submission.to_csv(submission_path, index=False)
print(f" Saved {len(submission)} predictions → {submission_path}")
print(submission.head())

**Kaggle Competition Submission**

In [ ]:
import os
from google.colab import userdata
#######################PART 5#################

COMPETITION_ID = "cis-515-vi-t-assignment"  # do not change this, it is the competition id
SUBMISSION_MSG = f"ViT fine-tune — {STUDENT_NAME}"

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"]      = userdata.get("KAGGLE_API_TOKEN")

!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f "{submission_path}" \
    -m "{SUBMISSION_MSG}"

print(f"\nSubmitted: Check your score at:")
print(f"   https://www.kaggle.com/competitions/{COMPETITION_ID}/leaderboard")

In [ ]:
# Run this to see what the student CSV looks like
print(submission.head())
print(submission.columns.tolist())